# Inspect Silver NYC TLC

Notebook para visualizar a Silver Delta da NYC TLC 2025.

Por padrao, este notebook tenta abrir primeiro a amostra `dev`. Se ela nao existir, tenta abrir a Silver oficial.

In [27]:
from v2.config.paths import DELTA_ROOT, nyc_tlc_silver_dir

official_path = nyc_tlc_silver_dir(2025)
dev_path = DELTA_ROOT / "dev" / "silver" / "nyc_tlc" / "yellow_sample" / "2025_01"

path = dev_path if (dev_path / "_delta_log").exists() else official_path
delta_log = path / "_delta_log"

print(f"Official path: {official_path}")
print(f"Dev path     : {dev_path}")
print(f"Path: {path}")
print(f"Path exists: {path.exists()}")
print(f"Delta log exists: {delta_log.exists()}")

if not delta_log.exists():
    raise FileNotFoundError(
        "Tabela Delta Silver nao encontrada. Rode primeiro a Silver dev ou a Silver oficial."
    )

Official path: /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/silver/nyc_tlc/yellow/2025
Dev path     : /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/silver/nyc_tlc/yellow_sample/2025_01
Path: /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/silver/nyc_tlc/yellow_sample/2025_01
Path exists: True
Delta log exists: True


In [28]:
from v2.config.spark import create_spark

spark = create_spark("NotebookInspectSilverNYCTLC")

In [29]:
df = spark.read.format("delta").load(str(path))
df.printSchema()

root
 |-- id_vendedor: integer (nullable = true)
 |-- data_hora_partida: timestamp_ntz (nullable = true)
 |-- data_hora_chegada: timestamp_ntz (nullable = true)
 |-- qtd_passageiros: integer (nullable = true)
 |-- distancia_milhas: double (nullable = true)
 |-- id_tarifa: integer (nullable = true)
 |-- flag_armazenado_e_enviado: string (nullable = true)
 |-- id_local_partida: integer (nullable = true)
 |-- id_local_chegada: integer (nullable = true)
 |-- tipo_pagamento: integer (nullable = true)
 |-- valor_tarifa: double (nullable = true)
 |-- taxa_extra: double (nullable = true)
 |-- taxa_mta_fixa: double (nullable = true)
 |-- gorjeta: double (nullable = true)
 |-- valor_pedagios: double (nullable = true)
 |-- sobretaxa_melhoria: double (nullable = true)
 |-- valor_total: double (nullable = true)
 |-- sobretaxa_transito: double (nullable = true)
 |-- taxa_aeroporto: double (nullable = true)
 |-- taxa_congestionamento_cbd: double (nullable = true)
 |-- data_viagem: date (nullable = tr

In [30]:
df.show(10, truncate=False)

+-----------+-------------------+-------------------+---------------+----------------+---------+-------------------------+----------------+----------------+--------------+------------+----------+-------------+-------+--------------+------------------+-----------+------------------+--------------+-------------------------+-----------+----+---+--------+-------+------------+--------------+---------------+-------------+-----------+------------+---------------+------------+------------+--------------------+
|id_vendedor|data_hora_partida  |data_hora_chegada  |qtd_passageiros|distancia_milhas|id_tarifa|flag_armazenado_e_enviado|id_local_partida|id_local_chegada|tipo_pagamento|valor_tarifa|taxa_extra|taxa_mta_fixa|gorjeta|valor_pedagios|sobretaxa_melhoria|valor_total|sobretaxa_transito|taxa_aeroporto|taxa_congestionamento_cbd|data_viagem|ano |mes|mes_nome|dia_mes|hora_partida|dia_semana_num|dia_semana_nome|fim_de_semana|periodo_dia|horario_pico|duracao_minutos|distancia_km|valor_por_km|veloci

In [31]:
df.select(
    "id_vendedor",
    "data_hora_partida",
    "data_hora_chegada",
    "id_local_partida",
    "id_local_chegada",
    "distancia_milhas",
    "distancia_km",
    "valor_total",
    "valor_por_km",
    "data_viagem",
    "ano",
    "mes",
    "mes_nome",
    "dia_mes",
    "hora_partida",
    "dia_semana_num",
    "dia_semana_nome",
    "fim_de_semana",
    "periodo_dia",
    "horario_pico",
    "duracao_minutos",
    "velocidade_media_kmh",
).show(20, truncate=False)

+-----------+-------------------+-------------------+----------------+----------------+----------------+------------+-----------+------------+-----------+----+---+--------+-------+------------+--------------+---------------+-------------+-----------+------------+---------------+--------------------+
|id_vendedor|data_hora_partida  |data_hora_chegada  |id_local_partida|id_local_chegada|distancia_milhas|distancia_km|valor_total|valor_por_km|data_viagem|ano |mes|mes_nome|dia_mes|hora_partida|dia_semana_num|dia_semana_nome|fim_de_semana|periodo_dia|horario_pico|duracao_minutos|velocidade_media_kmh|
+-----------+-------------------+-------------------+----------------+----------------+----------------+------------+-----------+------------+-----------+----+---+--------+-------+------------+--------------+---------------+-------------+-----------+------------+---------------+--------------------+
|2          |2025-01-01 01:20:55|2025-01-01 01:49:24|68              |1               |16.3      

In [32]:
# Use limit antes de converter para Pandas.
df.limit(30).toPandas()

,id_vendedor,data_hora_partida,data_hora_chegada,qtd_passageiros,distancia_milhas,id_tarifa,flag_armazenado_e_enviado,id_local_partida,id_local_chegada,tipo_pagamento,...,hora_partida,dia_semana_num,dia_semana_nome,fim_de_semana,periodo_dia,horario_pico,duracao_minutos,distancia_km,valor_por_km,velocidade_media_kmh
0,2,2025-01-01 01:20:55,2025-01-01 01:49:24,2,16.30,3,N,68,1,2,...,1,4,quarta,False,madrugada,False,28.48,26.23,3.76,55.26
1,2,2025-01-01 02:45:05,2025-01-01 03:13:28,3,21.19,4,N,170,1,1,...,2,4,quarta,False,madrugada,False,28.38,34.10,4.94,72.09
2,2,2025-01-01 03:17:27,2025-01-01 03:44:23,1,16.13,3,N,158,1,2,...,3,4,quarta,False,madrugada,False,26.93,25.96,4.07,57.84
3,2,2025-01-01 03:31:00,2025-01-01 04:11:38,1,22.67,3,N,41,1,1,...,3,4,quarta,False,madrugada,False,40.63,36.48,3.65,53.87
4,2,2025-01-01 03:38:24,2025-01-01 04:03:21,1,17.17,4,N,186,1,3,...,3,4,quarta,False,madrugada,False,24.95,27.63,4.72,66.44
5,2,2025-01-01 03:38:57,2025-01-01 04:05:11,4,17.91,3,N,230,1,1,...,3,4,quarta,False,madrugada,False,26.23,28.82,4.54,65.92
6,2,2025-01-01 03:39:52,2025-01-01 04:10:36,1,17.33,3,N,170,1,2,...,3,4,quarta,False,madrugada,False,30.73,27.89,4.50,54.45
7,2,2025-01-01 03:41:06,2025-01-01 04:09:45,1,16.62,4,N,233,1,1,...,3,4,quarta,False,madrugada,False,28.65,26.75,5.57,56.02
8,2,2025-01-01 03:43:12,2025-01-01 04:08:00,4,16.94,3,N,48,1,1,...,3,4,quarta,False,madrugada,False,24.80,27.26,3.89,65.95
9,2,2025-01-01 04:07:23,2025-01-01 04:28:12,1,13.20,3,N,231,1,1,...,4,4,quarta,False,madrugada,False,20.82,21.24,4.82,61.21


In [33]:
# Na Silver oficial, count pode demorar. Na dev, deve ser leve.
df.count()

97086

In [34]:
spark.stop()